# 🤖 What is an Agent?
### Let's build our own agent, powered by already known LLM capabilities and the OpenAI SDK

## 0️⃣ 🎓 Foundations & Intuition

> **“A system powered by an LLM that can plan, reason, memorize, and act by leveraging external tools, APIs, or knowledge sources to accomplish user-defined tasks.”**
> — *OpenAI, LangChain, LlamaIndex, Anthropic docs (2023–2024)*

➡️ *These agent capabilities 🛠 are not magic 🪄 — they’re things we already know how to reproduce.*

- **🧠 Reason**
  Compare, infer, or transform information.
  *Example:* decide which city is hotter, or sort the top 5 cities by temperature.

- **⚡ Act**
  Invoke a tool or function to get information or perform an action — one time or multiple times.
  *Example:* call `get_weather("London")` and `get_weather("New York")`.

- **🗺️ Plan**
  Decide how many times to call a function, in what order, and with what arguments.
  *Example:* loop over 5 cities, calling the weather API for each.

- **💾 Memorize**
  Remember the results of function calls and context from previous questions.
  *Example:* recall that we already fetched London and New York weather to compare which city is colder.


### ‍🔍 How the Flow of an Agent looks like?

![agentic_flow](../images/part2_what_is_an_agent__agentic_flow.png)

- **🤖 LLM Call** — does the thinking: **🔍 Reasoning** + **📝 Planning** + decides whether to act.

- **🛠️ Tool Call** — if action is needed, invoke the tool/function and capture the result (or error).

- **🔁 Reasoning Loop** — repeat: *Reason → Plan → (Act if needed) → Memorize*, until the model can confidently answer or a stop condition is reached.

## 1️⃣ 🧩 Assembling the Agent - building on last lesson’s code

### 📜 Contract of an Agent

In [1]:
import os
from abc import ABC, abstractmethod

class ABCAgent(ABC):
    @abstractmethod
    def __init__(self): ## TODO add to the contract params that are needed
        """Initialize core state of MyAgent class: system prompt, API key, client, message history, and tool registry."""
        self._api_key = os.getenv("OPENAI_API_KEY")
        if not self._api_key:
            raise ValueError("API key not found. Please set the OPENAI_API_KEY environment variable.")
        self.client = ...
        self.model = ...
        self.messages = ...
        self.tools = ...

    @abstractmethod
    def run(self, input: str):
        """Execute the agent on the given input <instructions> and return the final output."""
        output: str = ...
        return output

class ABCTool(ABC):
    @abstractmethod
    def __init__(self): ## TODO add to the contract params that are needed
        self.name = ...
        self.impl = ...
        self.spec = ...

### 📝 Implementation of an Agent

In [2]:
class OpenAITool(ABCTool):
    def __init__(self, name, impl, spec):
        super().__init__()
        self.name = name
        self.impl = impl
        self.spec = spec

In [11]:
import json

class OpenAIAgent(ABCAgent):
    SYSTEM_PROMPT = "You are helpful assistant."
    def __init__(self, tools: list[ABCTool]):
        super().__init__()
        self._client = OpenAI()
        self._model = "gpt-4o"
        self._messages = [{"role": "system", "content": OpenAIAgent.SYSTEM_PROMPT}]
        self._tool_registry = {tool.name: tool for tool in tools}

    def run(self, input: str):
        self._messages.append({"role": "system", "content": input})

        while True:
            shot = self._client.chat.completions.create(
                model = self._model,
                messages = self._messages,
                functions = [tool.spec for tool in self._tool_registry.values()],
                function_call="auto"
            )

            response = shot.choices[0].message
            self._messages.append(response)

            if response.function_call is not None:
                function_name = response.function_call.name
                function_args = json.loads(response.function_call.arguments)

                if function_name in self._tool_registry.keys():
                    function_impl = self._tool_registry[function_name].impl
                    response = function_impl(**function_args)

                    self._messages.append({
                        "role": "function",
                        "name": function_name,
                        "content": response
                    })
                else:
                    raise ValueError(f"Function {function_name} has not been specified.")
            elif shot.choices[0].finish_reason == "stop":
                return response

    def export_history(self, path: str):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self._messages, f)

### 🗣️ Let's call our Agent

#### We need to prepare (in reverse order)

1. **🤖 An agent**
   Takes a clear `system_prompt` and a **list of tools** the agent can call.
   *Goal:* return only the final result and make one tool call at a time.

2. **🧰 A tool registry**
   A list/dict of tool objects, each with a **name**, a **spec**, and an **implementation**.
   *Goal:* fast lookup by name and consistent execution.

3. **🛠️ Function impl**
   The actual Python callables that do the work (e.g., `add(a, b) -> float`).
   *Goal:* small, pure, well-typed, with clear errors (e.g., division by zero).

4. **📐 Function spec**
   JSON Schema (or similar) describing each tool’s **parameters**, their **types**, and **required** fields.
   *Goal:* let the agent know exactly how to call each function.


#### **📐 Function spec**

In [12]:
get_weather__spec = {
    "name": "get_weather",
    "description": "Get the current temperature & weather for a city",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {
                "type": "string",
                "description": "The city",
            },
        },
        "required": ["city"]
    }
}

#### **🛠️ Function impl**

In [13]:
import requests
from dotenv import dotenv_values

def get_weather(city):
    secrets = dotenv_values("../.env")
    url = ("""https://api.openweathermap.org/data/2.5/weather?q={}&appid={}&units=metric"""
           .format(city, secrets["OPEN_WEATHER_API_KEY"]))
    response = requests.get(url)

    if response.status_code != 200:
        raise Exception("An error has occured, code: {}, reason: {}".format(response.status_code, response.reason))
    data_dict = response.json()

    return json.dumps({
        "city": data_dict['name'],
        "weather": data_dict['weather'][0]['description'],
        "temperature": data_dict['main']['temp'],
        "unit": "Celsius"
    })

#### **🧰 Tool registry**

In [14]:
weather_tool = OpenAITool(
    name="get_weather",
    impl=get_weather,
    spec=get_weather__spec
)

#### **🤖 Agent**

In [15]:
secrets = dotenv_values("../.env")
os.environ["OPENAI_API_KEY"] = secrets["OPENAI_API_KEY"]

agent = OpenAIAgent(
    tools=[weather_tool]
)

#### **🚀 Run the Agent**

In [16]:
output = agent.run("Where is the weather better right now, London or New York?")

In [17]:
print(output.content)

Right now, London has broken clouds with a temperature of 14.79°C, whereas New York has overcast clouds with a cooler temperature of 1.58°C. Based on the current conditions, London has slightly better weather.


## 2️⃣  Practice: Test Your Agent 🧪🔧

Now it’s time to use the knowledge you gained in the lab in practice.
Try to make the following tasks.

- **Exercise 1:** *Execute prompts from lab1 in agent setup* 📘
- **Exercise 2:** *Ask the agent for the current date and time. If it can’t answer, solve it using what you learned in Lab 2* ⏱️
- **Exercise 3:** *Refactor the agent for flexibility—custom system prompt & model, + save/load chat history to JSON* 🧩
- **Exercise 4:** *Make the agent backend-agnostic—allow swapping OpenAI for any other model* 🔌

### 2.1 ✍️ Exercise
*Execute prompts from lab1 in agent setup* 📘
>- *Where is the weather better right now, London or New York?* 🌍
>- *Which city is colder, and by how many degrees?* ❄️
>- *Give me the current weather in the five biggest cities in Europe. Sort the cities by temperature.* 🏙️


In [18]:
secrets = dotenv_values("../.env")
os.environ["OPENAI_API_KEY"] = secrets["OPENAI_API_KEY"]

agent = OpenAIAgent(
    tools=[weather_tool]
)

In [19]:
output = agent.run("Where is the weather better right now, London or New York?")
print(output.content)

Right now, the weather in London is "broken clouds" with a temperature of 14.79°C, while in New York, it's "overcast clouds" with a temperature of 1.58°C. Based on these conditions, London has milder and potentially more pleasant weather at this moment.


In [20]:
output = agent.run("Which city is colder, and by how many degrees?")
print(output.content)

New York is colder than London by 13.21 degrees Celsius.


In [21]:
output = agent.run("Give me the current weather in the five biggest cities in Europe. Sort the cities by temperature.")
print(output.content)

Here is the current weather in the five biggest cities in Europe, sorted by temperature:

1. **Istanbul**: 18.42°C, moderate rain
2. **London**: 14.79°C, broken clouds
3. **Paris**: 14.22°C, overcast clouds
4. **Berlin**: 9.57°C, broken clouds
5. **Moscow**: 5.74°C, overcast clouds


### 📌 Conclusion

> - Using agent **abstraction is useful**. It simplifies code.

### 2.2 ✍️ Exercise
*Ask an agent for current date and time. If the agent does not know - build a get_datetime_tool* ⏱️

In [22]:
secrets = dotenv_values("../.env")
os.environ["OPENAI_API_KEY"] = secrets["OPENAI_API_KEY"]

agent = OpenAIAgent(
    tools=[weather_tool]
)

In [23]:
output = agent.run("What time it is?")
print(output.content)

I'm unable to provide real-time information, including the current time. However, you can check the time using a device like a smartphone, computer, or watch.


In [24]:
get_datetime__spec = {
    "name": "get_datetime",
    "description": "Get the current date and time",
    "parameters": {
        "type": "object",
        "properties": {},
        "required": []
    }
}

In [25]:
from datetime import datetime
import json

def get_datetime():
    now = datetime.now()

    return json.dumps({
        "date": now.strftime("%Y-%m-%d"),
        "time": now.strftime("%H:%M:%S"),
        "iso": now.isoformat()
    })

In [26]:
datetime_tool = OpenAITool(
    name="get_datetime",
    impl=get_datetime,
    spec=get_datetime__spec
)

In [27]:
secrets = dotenv_values("../.env")
os.environ["OPENAI_API_KEY"] = secrets["OPENAI_API_KEY"]

agent = OpenAIAgent(
    tools=[datetime_tool]
)

In [28]:
output = agent.run("What time it is?")
print(output.content)

The current time is 14:21 (or 2:21 PM) on November 11, 2025.


### 2.3 ✍️ Exercise

*Refactor the agent for flexibility—custom system prompt & model, + save/load chat history to JSON* 🧩

**Hint:** Ask the LLM for missing Python code or quick guidelines when you get stuck. 🤖

In [31]:
from openai import OpenAI

class OpenAIAgent(ABCAgent):
    SYSTEM_PROMPT = "You are helpful assistant."
    def __init__(self, tools: list[ABCTool], model="gpt-4o", initial_messages: list[dict] | None = None):
        super().__init__()
        self._client = OpenAI()
        self._model = model
        self._messages = initial_messages or [{"role": "system", "content": self.SYSTEM_PROMPT}]
        self._tool_registry = {tool.name: tool for tool in tools}

    def run(self, input: str):
        self._messages.append({"role": "system", "content": input})

        while True:
            shot = self._client.chat.completions.create(
                model = self._model,
                messages = self._messages,
                functions = [tool.spec for tool in self._tool_registry.values()],
                function_call="auto"
            )

            response = shot.choices[0].message
            self._messages.append(response)

            if response.function_call is not None:
                function_name = response.function_call.name
                function_args = json.loads(response.function_call.arguments)

                if function_name in self._tool_registry.keys():
                    function_impl = self._tool_registry[function_name].impl
                    response = function_impl(**function_args)

                    self._messages.append({
                        "role": "function",
                        "name": function_name,
                        "content": response
                    })
                else:
                    raise ValueError(f"Function {function_name} has not been specified.")
            elif shot.choices[0].finish_reason == "stop":
                return response

    def export_history(self, path: str):
        with open(path, "w", encoding="utf-8") as f:
            json.dump(self._messages, f)

    @classmethod
    def from_history_file(cls, history_file_path: str, tools: list[ABCTool], model: str = "gpt-4o"):
        with open(history_file_path, "r", encoding="utf-8") as f:
            messages = json.load(f)
        return cls(tools=tools, model=model, initial_messages=messages)

### 2.4 ✍️ Exercise

*Make the agent backend-agnostic—allow swapping OpenAI for any other model* 🔌

**Hint 1:** Use the strategy pattern with a Python `Protocol` to define a tiny, universal backend interface. 🧱
**Hint 2:** Begin with a lightweight `EchoBackend` to test your setup without hitting any real API. 🪞



In [37]:
from __future__ import annotations
from typing import Protocol, Dict

class LLMBackend(Protocol):
    def run(self, input: str) -> dict:
        ...

class Agent:
    def __init__(self, backend: LLMBackend):
        self._backend = backend

    def ask(self, text: str) -> Dict:
        return self._backend.run(text)

In [34]:
class OpenAIBackend:
    SYSTEM_PROMPT = "You are helpful assistant."
    def __init__(self, tools: list[ABCTool]):
        self._client = OpenAI()
        self._model = "gpt-4o"
        self._messages = [{"role": "system", "content": OpenAIBackend.SYSTEM_PROMPT}]
        self._tool_registry = {tool.name: tool for tool in tools}

    def run(self, message: str) -> dict:
        ... ## TODO we know it from the previous implementations

class EchoBackend:
    SYSTEM_PROMPT = "You are helpful assistant."
    def __init__(self):
        self._messages = [{"role": "system", "content": EchoBackend.SYSTEM_PROMPT}]

    def run(self, message: str) -> dict:
        self._messages.append({"role": "user", "content": message})
        reply = {
            "role": "assistant",
            "content": f"(echo) {message}",
            "function_call": None,
            "finish_reason": "stop",
        }
        self._messages.append(reply)
        return reply

In [35]:
agent = Agent(backend=EchoBackend())

In [36]:
output = agent.ask("What time it is?")
print(output['content'])

(echo) What time it is?


### 📌 Conclusion

> - Each extension increases framework complexity and the effort required to maintain it.
> - The above (extension building & maintenance) **can slow experimentation**.
> - Rapid LLM changes can **push you to maintain the framework** more than the project itself.

## 3️⃣ Conclusions & Summary 📝

In Part 2, we explored *agents* — systems that can **reason, act, plan, and remember**. We defined what an agent is, built our own version in Python, and watched it behave step by step.

We covered:

- 🧠 **Agent Capabilities** — how agents reason, plan, memorize and act.
- 🧩 **Python Abstractions** — a clean agent + tool contract, then both implemented from scratch.
- 🧪 **Practical Exercises** — build & use custom tools, pros and cons of abstraction.

From this work, a few lessons stood out:

- ✅ Agent abstractions **simplify code** and make behavior clearer, but..
- ⚠️ Every extension adds **complexity and maintenance overhead**.
- 🐌 More structure can **slow experimentation**.
- ⏳ LLMs evolve fast, and custom agents can quickly turn into **mini-frameworks you must maintain**.

➡️ This naturally leads to our next step:
**Part 3 (Frameworks)** — using established agent frameworks to stay productive without reinventing everything ourselves.
